# Model 3 — Gated Segmentation Network
### CS 6140 — Copy-Move Forgery Detection in Biomedical Scientific Images

**Authors:** Seshadri Veeraraghavan Vidyalakshmi, Jaisweta Naarrayanan  
**Hardware:** Kaggle T4 GPU  

---

## What this notebook does

This notebook trains a two-stage deep learning pipeline to detect copy-move forgeries:

**Stage 1 — Gate Classifier (`EfficientNet-B0`)**  
A lightweight binary classifier that answers one question: *is this image forged or authentic?*  
If authentic, we output an empty mask and stop. No need to run the expensive segmenter.

**Stage 2 — Segmentation Network (`ResNet-34` encoder + `UNet` decoder)**  
Only runs on images the gate flagged as forged. Outputs a pixel-level binary mask  
showing exactly which pixels were copy-moved.

**Why this two-stage design?**  
Running segmentation on every image — including authentic ones — introduces false positives  
and wastes compute. The gate acts as a filter: only images that need a mask get one.

---

## Notebook structure

| Section | What it covers |
|---|---|
| 1. Setup | Install deps, imports, paths, device |
| 2. Data utilities | File collection, train/val/test split, checkpointing |
| 3. Gate — dataset & model | EfficientNet-B0, transforms, training loop |
| 4. Gate — training | Run training with early stopping |
| 5. Gate — evaluation | Test set metrics, confusion matrix, curves |
| 6. Segmenter — dataset & model | UNet, BCE+Dice loss, SE attention |
| 7. Segmenter — training | Run training with early stopping |
| 8. Segmenter — evaluation | Test Dice/IoU, prediction visualisations |
| 9. Download outputs | Links to all saved files |

**To run:** Execute cells top to bottom.  
**To resume:** Set `GATE_RESUME = True` or `SEG_RESUME = True` in the config cells.

---
## 1. Setup

In [1]:
# Install libraries not available by default on Kaggle
# segmentation-models-pytorch: provides pre-built UNet/ResNet architectures
# albumentations: fast, mask-aware image augmentation
# tqdm: progress bars for training loops
!pip install segmentation-models-pytorch albumentations tqdm -q
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:0000:01
Dependencies installed.


In [2]:
import os
import time
import logging
from pathlib import Path

import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive backend which works on Kaggle servers
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import models, transforms

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, ConfusionMatrixDisplay,
)
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s')
log = logging.getLogger(__name__)
print('All imports successful.')

All imports successful.


In [17]:
# Paths
# Update DATA_DIR to match where your dataset is mounted.
# After attaching the dataset in Kaggle, run os.listdir('/kaggle/input')
# to find the exact folder name.
DATA_DIR      = Path('/kaggle/input/datasets/sesha28/scientific-image-forgery-detection/data')
AUTHENTIC_DIR = DATA_DIR / 'train_images' / 'authentic'
FORGED_DIR    = DATA_DIR / 'train_images' / 'forged'
MASK_DIR      = DATA_DIR / 'train_masks'
SUPP_AUTH     = DATA_DIR / 'supplemental_images' / 'authentic'
SUPP_FORG     = DATA_DIR / 'supplemental_images' / 'forged'
OUTPUT_DIR    = Path('/kaggle/working')   # writable; download from here after training
OUTPUT_DIR.mkdir(exist_ok=True)

# Shared constants
VALID_EXT   = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
IMAGE_SIZE  = 512    # all images are resized to this before entering any model
RANDOM_SEED = 42

# ImageNet normalisation — both models use pretrained encoders trained on ImageNet
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# Device setup
def get_device():
    """Use CUDA (Kaggle GPU) if available, otherwise use CPU."""
    if torch.cuda.is_available():
        log.info('Device: CUDA'); return torch.device('cuda')
    log.info('Device: CPU');  return torch.device('cpu')

DEVICE = get_device()

# Verify all expected directories exist before proceeding
for name, path in [('DATA_DIR', DATA_DIR), ('AUTHENTIC_DIR', AUTHENTIC_DIR),
                   ('FORGED_DIR', FORGED_DIR), ('MASK_DIR', MASK_DIR)]:
    status = 'ok' if path.exists() else 'err:  NOT FOUND'
    print(f'{name:15s} {status}  ({path})')

2026-04-22 19:10:45,905  Device: CUDA


DATA_DIR        ok  (/kaggle/input/datasets/sesha28/scientific-image-forgery-detection/data)
AUTHENTIC_DIR   ok  (/kaggle/input/datasets/sesha28/scientific-image-forgery-detection/data/train_images/authentic)
FORGED_DIR      ok  (/kaggle/input/datasets/sesha28/scientific-image-forgery-detection/data/train_images/forged)
MASK_DIR        ok  (/kaggle/input/datasets/sesha28/scientific-image-forgery-detection/data/train_masks)


---
## 2. Data Utilities

These helpers are shared between the gate and segmenter — both use the same images,
just with different targets (class label vs. pixel mask).

In [18]:
def collect_pairs(authentic_dir, forged_dir, samples_per_class=None, seed=RANDOM_SEED):
    """
    Go through authentic/ and forged/ directories and return a shuffled list of
    (image_path, label) pairs. Label 0 = authentic, 1 = forged.

    Also includes supplemental_images/ if those directories exist.
    Optionally limits to `samples_per_class` images per class.
    """
    auth = sorted(f for f in authentic_dir.glob('*') if f.suffix.lower() in VALID_EXT)
    forg = sorted(f for f in forged_dir.glob('*')    if f.suffix.lower() in VALID_EXT)

    if SUPP_AUTH.exists():
        auth += sorted(f for f in SUPP_AUTH.glob('*') if f.suffix.lower() in VALID_EXT)
    if SUPP_FORG.exists():
        forg += sorted(f for f in SUPP_FORG.glob('*') if f.suffix.lower() in VALID_EXT)

    log.info(f'Found  {len(auth)} authentic  |  {len(forg)} forged')

    rng = np.random.default_rng(seed)
    if samples_per_class:
        auth = list(rng.choice(auth, size=min(samples_per_class, len(auth)), replace=False))
        forg = list(rng.choice(forg, size=min(samples_per_class, len(forg)), replace=False))

    pairs = [(p, 0) for p in auth] + [(p, 1) for p in forg]
    rng.shuffle(pairs)
    return pairs


def split_pairs(pairs, seed=RANDOM_SEED):
    """
    70 / 15 / 15 stratified split into train / val / test.
    """
    labels      = [p[1] for p in pairs]
    train, temp = train_test_split(pairs, test_size=0.30, stratify=labels, random_state=seed)
    temp_labels = [p[1] for p in temp]
    val, test   = train_test_split(temp,  test_size=0.50, stratify=temp_labels, random_state=seed)
    log.info(f'Split  train={len(train)}  val={len(val)}  test={len(test)}')
    return train, val, test


def save_checkpoint(state, path):
    """Persist model + optimiser + scheduler state so training can resume."""
    torch.save(state, path)
    log.info(f'Checkpoint saved to {path}')


def plot_confusion_matrix(y_true, y_pred, title, path):
    """Save a labelled confusion matrix as a PNG."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=['Authentic', 'Forged']).plot(ax=ax)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=150); plt.close()
    log.info(f'Confusion matrix saved to {path}')


print('Data utilities ready.')

Data utilities ready.


---
## 3. Stage 1 — Gate Classifier

### Why EfficientNet-B0?
EfficientNet-B0 is small and fast (5.3M parameters) while still benefiting from
deep convolutional feature representations. We use pretrained ImageNet weights and
only fine-tune the classifier head — this is called *transfer learning*, and it
works well even when the target domain (biomedical images) differs from the source.

### Architecture change
The original EfficientNet-B0 ends in a 1000-class linear layer (for ImageNet).
We replace it with `Dropout(0.5) to Linear(in_features, 1)` — a single neuron
that outputs a raw logit. We chose Dropout=0.5 (higher than the default 0.2)
because we observed overfitting starting around epoch 4-5 in early experiments.

In [19]:
# Gate config
GATE_RESUME            = False  # set True to resume from gate_checkpoint.pth
GATE_SAMPLES_PER_CLASS = 1200   # images per class; increase for better generalisation
GATE_BATCH_SIZE        = 32     # larger batch = smoother gradients on GPU
GATE_NUM_EPOCHS        = 20
GATE_LR                = 1e-4   # conservative — avoids destabilising pretrained weights
GATE_WEIGHT_DECAY      = 1e-4   # L2 regularisation via AdamW
GATE_PATIENCE          = 5      # stop if val F1 doesn't improve for this many epochs

GATE_CHECKPOINT = OUTPUT_DIR / 'gate_checkpoint.pth'
GATE_BEST       = OUTPUT_DIR / 'gate_best.pth'
print('Gate config ready.')

Gate config ready.


In [20]:
class GateDataset(Dataset):
    """
    Loads images for binary classification.
    Returns (image_tensor, label) where label is 0.0 or 1.0 (float for BCEWithLogitsLoss).
    """
    def __init__(self, pairs, transform=None, image_size=IMAGE_SIZE):
        self.pairs      = pairs
        self.transform  = transform
        self.image_size = image_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        path, label = self.pairs[idx]
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.image_size, self.image_size))
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)


def gate_transforms(train):
    """
    Training augmentation: flip, rotate, colour jitter.
    These are safe for forgery detection — forgeries don't depend on orientation.
    Validation/test: just resize + normalise (no randomness).
    """
    ops = [transforms.ToPILImage()]
    if train:
        ops += [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ]
    ops += [
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
    return transforms.Compose(ops)


def build_gate_model():
    """
    EfficientNet-B0 with a custom binary classification head.
    Using pretrained=True means we start from weights already good at
    recognising visual patterns — we just redirect that ability to forgery.
    """
    model   = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_feat = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),    # higher dropout reduces overfitting on small datasets
        nn.Linear(in_feat, 1) # single logit -> sigmoid -> P(forged)
    )
    return model


print('Gate dataset and model defined.')

Gate dataset and model defined.


In [21]:
def gate_train_one_epoch(model, loader, optimizer, criterion, device, epoch, total):
    """Run one training epoch. tqdm shows live loss + accuracy per batch."""
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    pbar = tqdm(loader, desc=f'Gate epoch [{epoch+1:02d}/{total}]', ncols=100)

    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        correct    += ((torch.sigmoid(logits) >= 0.5).float() == labels).sum().item()
        n          += imgs.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{correct/n:.4f}')

    return total_loss / n, correct / n


@torch.no_grad()
def gate_evaluate(model, loader, criterion, device):
    """
    Evaluate gate on val or test set.
    Returns loss, accuracy, F1, ROC-AUC, and raw predictions + labels
    (needed for the confusion matrix and classification report).
    """
    model.eval()
    total_loss                       = 0.0
    all_probs, all_preds, all_labels = [], [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        logits       = model(imgs)
        total_loss  += criterion(logits, labels).item() * imgs.size(0)
        probs        = torch.sigmoid(logits).cpu().numpy().flatten()
        all_probs.extend(probs)
        all_preds.extend((probs >= 0.5).astype(int))
        all_labels.extend(labels.cpu().numpy().flatten().astype(int))

    n   = len(all_labels)
    f1  = f1_score(all_labels, all_preds, zero_division=0)
    auc = roc_auc_score(all_labels, all_probs)
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / n
    return total_loss / n, acc, f1, auc, all_preds, all_labels


print('Gate train/eval functions ready.')

Gate train/eval functions ready.


---
## 4. Stage 1 — Train Gate

Training details:
- **Optimizer:** AdamW — Adam with decoupled weight decay, better regularisation than vanilla Adam
- **LR Schedule:** Cosine annealing — smoothly decays LR from `GATE_LR` to near zero
- **Early stopping:** Halts training if val F1 doesn't improve for `GATE_PATIENCE` epochs
- **Checkpointing:** Saves every epoch so you can resume with `GATE_RESUME = True`

**Expected runtime:** ~10-15 minutes on Kaggle T4

In [22]:
# Build dataloaders
gate_pairs = collect_pairs(AUTHENTIC_DIR, FORGED_DIR, GATE_SAMPLES_PER_CLASS)
train_pairs, val_pairs, test_pairs = split_pairs(gate_pairs)

# num_workers=0 avoids multiprocessing deadlocks in Kaggle notebook sessions
train_dl = DataLoader(GateDataset(train_pairs, gate_transforms(True)),
                      batch_size=GATE_BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_dl   = DataLoader(GateDataset(val_pairs,   gate_transforms(False)),
                      batch_size=GATE_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_dl  = DataLoader(GateDataset(test_pairs,  gate_transforms(False)),
                      batch_size=GATE_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

# Build model, loss, optimiser, scheduler
gate_model     = build_gate_model().to(DEVICE)
gate_criterion = nn.BCEWithLogitsLoss()  # numerically stable sigmoid + BCE in one op
gate_optimizer = AdamW(gate_model.parameters(), lr=GATE_LR, weight_decay=GATE_WEIGHT_DECAY)
gate_scheduler = CosineAnnealingLR(gate_optimizer, T_max=GATE_NUM_EPOCHS, eta_min=1e-6)

# Initialise tracking variables
start_epoch, best_val_f1, no_improve   = 0, 0.0, 0
gate_train_losses, gate_val_losses, gate_val_f1s = [], [], []

# Resume from checkpoint if requested
if GATE_RESUME and GATE_CHECKPOINT.exists():
    ckpt = torch.load(GATE_CHECKPOINT, map_location='cpu')
    gate_model.load_state_dict(ckpt['model_state'])
    gate_optimizer.load_state_dict(ckpt['optimizer_state'])
    gate_scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch  = ckpt['epoch']
    best_val_f1  = ckpt['best_metric']
    log.info(f'Resumed from epoch {start_epoch}  (best val F1: {best_val_f1:.4f})')
elif GATE_RESUME:
    log.warning('GATE_RESUME=True but no checkpoint found. Starting from scratch.')

# Training loop
log.info('Starting gate training …')
for epoch in range(start_epoch, GATE_NUM_EPOCHS):
    t0 = time.time()

    train_loss, train_acc = gate_train_one_epoch(
        gate_model, train_dl, gate_optimizer, gate_criterion, DEVICE, epoch, GATE_NUM_EPOCHS)
    val_loss, val_acc, val_f1, val_auc, _, _ = gate_evaluate(
        gate_model, val_dl, gate_criterion, DEVICE)
    gate_scheduler.step()

    gate_train_losses.append(train_loss)
    gate_val_losses.append(val_loss)
    gate_val_f1s.append(val_f1)

    log.info(
        f'Epoch [{epoch+1:02d}/{GATE_NUM_EPOCHS}]  '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f}  '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}  '
        f'val_F1={val_f1:.4f} val_AUC={val_auc:.4f}  '
        f'({time.time()-t0:.1f}s)'
    )

    # Save checkpoint every epoch — allows resuming if interrupted
    save_checkpoint({
        'epoch':           epoch + 1,
        'model_state':     gate_model.state_dict(),
        'optimizer_state': gate_optimizer.state_dict(),
        'scheduler_state': gate_scheduler.state_dict(),
        'best_metric':     best_val_f1,
    }, GATE_CHECKPOINT)

    # Save best model separately (use for eval)
    if val_f1 > best_val_f1:
        best_val_f1, no_improve = val_f1, 0
        torch.save(gate_model.state_dict(), GATE_BEST)
        log.info(f' New best val F1: {best_val_f1:.4f} saved to {GATE_BEST}')
    else:
        no_improve += 1
        log.info(f' No improvement ({no_improve}/{GATE_PATIENCE})')

    if no_improve >= GATE_PATIENCE:
        log.info(f'Early stopping — no improvement for {GATE_PATIENCE} epochs.')
        break

log.info('Gate training complete.')

2026-04-22 19:11:03,130  Found  2377 authentic  |  2751 forged
2026-04-22 19:11:03,148  Split  train=1680  val=360  test=360
2026-04-22 19:11:03,300  Starting gate training …
Gate epoch [01/20]: 100%|██████████████████| 53/53 [02:00<00:00,  2.28s/it, acc=0.5125, loss=0.6888]
2026-04-22 19:13:19,258  Epoch [01/20]  train_loss=0.7007 train_acc=0.5125  val_loss=0.7017 val_acc=0.4833  val_F1=0.5053 val_AUC=0.5070  (136.0s)
2026-04-22 19:13:19,420  Checkpoint saved to /kaggle/working/gate_checkpoint.pth
2026-04-22 19:13:19,478   New best val F1: 0.5053 saved to /kaggle/working/gate_best.pth
Gate epoch [02/20]: 100%|██████████████████| 53/53 [01:57<00:00,  2.22s/it, acc=0.5446, loss=0.7076]
2026-04-22 19:15:31,887  Epoch [02/20]  train_loss=0.6841 train_acc=0.5446  val_loss=0.6974 val_acc=0.4972  val_F1=0.5617 val_AUC=0.5166  (132.4s)
2026-04-22 19:15:32,051  Checkpoint saved to /kaggle/working/gate_checkpoint.pth
2026-04-22 19:15:32,115   New best val F1: 0.5617 saved to /kaggle/working/gat

---
## 5. Stage 1 — Gate Evaluation

In [23]:
# Loss and F1 curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(gate_train_losses, label='Train Loss')
ax1.plot(gate_val_losses,   label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Gate — Loss Curves'); ax1.legend()
ax2.plot(gate_val_f1s, label='Val F1', color='green')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1')
ax2.set_title('Gate — Validation F1'); ax2.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gate_training_curves.png', dpi=150)
plt.show()

# Load best weights and evaluate on held-out test set
gate_model.load_state_dict(torch.load(GATE_BEST, map_location=DEVICE))
_, test_acc, test_f1, test_auc, test_preds, test_labels = gate_evaluate(
    gate_model, test_dl, gate_criterion, DEVICE)

print('\n' + '-'*60)
print('GATE — FINAL TEST SET RESULTS')
print(f'  Accuracy : {test_acc:.4f}')
print(f'  F1       : {test_f1:.4f}')
print(f'  ROC-AUC  : {test_auc:.4f}')
print('\nClassification Report:')
print(classification_report(test_labels, test_preds, target_names=['Authentic', 'Forged']))

plot_confusion_matrix(
    test_labels, test_preds,
    title='Confusion Matrix — Gate (EfficientNet-B0) — Test Set',
    path=OUTPUT_DIR / 'gate_confusion_matrix.png'
)

# Display the confusion matrix inline
plt.figure(figsize=(5, 4))
plt.imshow(plt.imread(str(OUTPUT_DIR / 'gate_confusion_matrix.png')))
plt.axis('off'); plt.show()


------------------------------------------------------------
GATE — FINAL TEST SET RESULTS
  Accuracy : 0.7194
  F1       : 0.7187
  ROC-AUC  : 0.8263

Classification Report:
              precision    recall  f1-score   support

   Authentic       0.72      0.72      0.72       180
      Forged       0.72      0.72      0.72       180

    accuracy                           0.72       360
   macro avg       0.72      0.72      0.72       360
weighted avg       0.72      0.72      0.72       360



2026-04-22 20:00:34,858  Confusion matrix saved to /kaggle/working/gate_confusion_matrix.png


---
## 6. Stage 2 — Segmenter

### Architecture: ResNet-34 encoder + UNet decoder + scSE attention

**UNet** is an encoder-decoder architecture with skip connections. The encoder
progressively downsamples the image to learn abstract features. The decoder
upsamples back to the original resolution. Skip connections pass feature maps
from the encoder directly to the decoder — this preserves spatial detail that
would otherwise be lost during downsampling.

**ResNet-34 encoder** gives us a strong pretrained feature extractor with
residual connections — stable training, good at hierarchical visual features.

**scSE (Squeeze-and-Excitation) attention** is added to each decoder block.
It has two components:
- *Channel SE*: learns which feature channels matter most and rescales them
- *Spatial SE*: learns which spatial locations to focus on

This is especially useful here because forged regions can be very small and
would otherwise be drowned out by background features.

### Loss: BCE + Dice

Forged pixels make up only 2-5% of each image. A model that predicts all
background would get 95%+ accuracy but be completely useless.

- **BCE** (Binary Cross-Entropy): penalises wrong pixel predictions, provides strong gradients
- **Dice**: directly measures region overlap — forces the model to actually cover forged regions

Without Dice, the model learns to predict background everywhere. Without BCE,
training is unstable early on. We use 50/50 weighting.

In [24]:
# Segmenter config 
SEG_RESUME       = False  # set True to resume from segmenter_checkpoint.pth
SEG_BATCH_SIZE   = 16     # reduce to 8 if you get out-of-memory errors
SEG_NUM_EPOCHS   = 30
SEG_LR           = 3e-4
SEG_WEIGHT_DECAY = 1e-4
SEG_PATIENCE     = 7      # segmenter needs more patience — harder task than gate
BCE_WEIGHT       = 0.5
DICE_WEIGHT      = 0.5

SEG_CHECKPOINT = OUTPUT_DIR / 'segmenter_checkpoint.pth'
SEG_BEST       = OUTPUT_DIR / 'segmenter_best.pth'
print('Segmenter config ready.')

Segmenter config ready.


In [25]:
class SegDataset(Dataset):
    """
    Loads images and their binary segmentation masks.

    Mask loading order:
      1. <image_stem>.npy  (preferred — stored as (1, H, W) uint8)
      2. <image_stem>.png  (fallback)
      3. All-zero mask     (for authentic images with no mask file)

    We use INTER_NEAREST when resizing masks because bilinear interpolation
    would create intermediate values between 0 and 1 in a binary mask,
    corrupting the ground truth labels.
    """
    def __init__(self, pairs, mask_dir, transform=None, image_size=IMAGE_SIZE):
        self.pairs      = pairs
        self.mask_dir   = mask_dir
        self.transform  = transform
        self.image_size = image_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, _ = self.pairs[idx]

        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.image_size, self.image_size))

        mask_npy = self.mask_dir / (img_path.stem + '.npy')
        mask_png = self.mask_dir / (img_path.stem + '.png')

        if mask_npy.exists():
            # Masks are stored as (1, H, W) uint8 — squeeze before converting
            mask = np.squeeze(np.load(str(mask_npy))).astype(np.float32)
        elif mask_png.exists():
            mask = (cv2.imread(str(mask_png), cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)
        else:
            mask = np.zeros((self.image_size, self.image_size), dtype=np.float32)

        # Guard against unexpected shapes from loading quirks
        if mask.ndim != 2 or mask.shape[0] == 0 or mask.shape[1] == 0:
            mask = np.zeros((self.image_size, self.image_size), dtype=np.float32)

        mask = cv2.resize(mask, (self.image_size, self.image_size),
                          interpolation=cv2.INTER_NEAREST)
        mask = (mask > 0.5).astype(np.float32)

        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out['image']
            mask = out['mask'].unsqueeze(0)   # (H,W) to (1,H,W) to match model output shape
        else:
            img  = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0
            mask = torch.from_numpy(mask).unsqueeze(0)

        return img, mask


def seg_transforms(train):
    """
    Albumentations applies transforms consistently to both image and mask.
    This is important — if we flip the image we must also flip the mask.
    """
    if train:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.GaussNoise(p=0.3),              # simulate sensor noise
            A.Blur(blur_limit=3, p=0.2),      # simulate focus variation
            A.RandomBrightnessContrast(p=0.3),
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])


def build_segmenter():
    """
    ResNet-34/UNet with Squeeze-and-Excitation attention in the decoder.
    activation=None means the model outputs raw logits — sigmoid is applied
    inside the loss function for numerical stability.
    """
    return smp.Unet(
        encoder_name           = 'resnet34',
        encoder_weights        = 'imagenet',
        in_channels            = 3,
        classes                = 1,
        activation             = None,
        decoder_attention_type = 'scse',  # Squeeze-and-Excitation in decoder blocks
    )


class BCEDiceLoss(nn.Module):
    """
    BCE + Dice combined loss.
    - BCE: pixel-wise correctness, strong gradients
    - Dice: region overlap, critical for class imbalance (2-5% forged pixels)
    """
    def __init__(self, bce_w=0.5, dice_w=0.5, smooth=1e-6):
        super().__init__()
        self.bce_w  = bce_w
        self.dice_w = dice_w
        self.smooth = smooth
        self.bce    = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        probs  = torch.sigmoid(logits)
        flat_p = probs.view(-1); flat_t = targets.view(-1)
        inter  = (flat_p * flat_t).sum()
        dice   = 1 - (2*inter + self.smooth) / (flat_p.sum() + flat_t.sum() + self.smooth)
        return self.bce_w * self.bce(logits, targets) + self.dice_w * dice


def compute_seg_metrics(logits, targets, threshold=0.5):
    """
    Dice, IoU, precision, recall on a batch.
    Pixel accuracy is excluded — it's misleading under class imbalance.
    """
    preds  = (torch.sigmoid(logits) >= threshold).float()
    smooth = 1e-6
    inter  = (preds * targets).sum()
    union  = preds.sum() + targets.sum()
    tp     = inter
    fp     = (preds * (1 - targets)).sum()
    fn     = ((1 - preds) * targets).sum()
    return {
        'dice':      ((2*inter + smooth) / (union + smooth)).item(),
        'iou':       ((inter + smooth)   / (union - inter + smooth)).item(),
        'precision': ((tp + smooth) / (tp + fp + smooth)).item(),
        'recall':    ((tp + smooth) / (tp + fn + smooth)).item(),
    }


print('Segmenter components defined.')

Segmenter components defined.


In [26]:
def seg_train_one_epoch(model, loader, optimizer, criterion, device, epoch, total):
    """Run one training epoch. Returns (avg_loss, metrics_dict)."""
    model.train()
    total_loss = 0.0
    metrics    = {'dice': 0.0, 'iou': 0.0, 'precision': 0.0, 'recall': 0.0}
    pbar = tqdm(loader, desc=f'Seg epoch [{epoch+1:02d}/{total}]', ncols=110)

    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, masks)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        m = compute_seg_metrics(logits.detach(), masks)
        for k in metrics: metrics[k] += m[k] * imgs.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}', dice=f'{m["dice"]:.4f}')

    n = len(loader.dataset)
    return total_loss / n, {k: v/n for k, v in metrics.items()}


@torch.no_grad()
def seg_evaluate(model, loader, criterion, device):
    """Evaluate segmenter on val or test set. Returns (avg_loss, metrics_dict)."""
    model.eval()
    total_loss = 0.0
    metrics    = {'dice': 0.0, 'iou': 0.0, 'precision': 0.0, 'recall': 0.0}

    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits      = model(imgs)
        total_loss += criterion(logits, masks).item() * imgs.size(0)
        m = compute_seg_metrics(logits, masks)
        for k in metrics: metrics[k] += m[k] * imgs.size(0)

    n = len(loader.dataset)
    return total_loss / n, {k: v/n for k, v in metrics.items()}


print('Segmenter train/eval functions ready.')

Segmenter train/eval functions ready.


---
## 7. Stage 2 — Train Segmenter

**Expected runtime:** ~3-4 hours on Kaggle T4 for 30 epochs  
**Tip:** If the cell deadlocks mid-epoch, set `SEG_RESUME = True` and re-run — it picks up from the last saved checkpoint.

In [27]:
# Build dataloaders 
seg_pairs = collect_pairs(AUTHENTIC_DIR, FORGED_DIR)
seg_train, seg_val, seg_test = split_pairs(seg_pairs)

seg_train_dl = DataLoader(SegDataset(seg_train, MASK_DIR, seg_transforms(True),  image_size=512),
                          batch_size=SEG_BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
seg_val_dl   = DataLoader(SegDataset(seg_val,   MASK_DIR, seg_transforms(False), image_size=512),
                          batch_size=SEG_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
seg_test_dl  = DataLoader(SegDataset(seg_test,  MASK_DIR, seg_transforms(False), image_size=512),
                          batch_size=SEG_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

# Build model, loss, optimiser, scheduler
seg_model     = build_segmenter().to(DEVICE)
seg_criterion = BCEDiceLoss(bce_w=BCE_WEIGHT, dice_w=DICE_WEIGHT)
seg_optimizer = AdamW(seg_model.parameters(), lr=SEG_LR, weight_decay=SEG_WEIGHT_DECAY)
seg_scheduler = CosineAnnealingLR(seg_optimizer, T_max=SEG_NUM_EPOCHS, eta_min=1e-6)

# Initialise tracking variables
start_epoch, best_val_dice, no_improve        = 0, 0.0, 0
seg_train_losses, seg_val_losses, seg_val_dices = [], [], []

# Resume from checkpoint if requested
if SEG_RESUME and SEG_CHECKPOINT.exists():
    ckpt = torch.load(SEG_CHECKPOINT, map_location='cpu')
    seg_model.load_state_dict(ckpt['model_state'])
    seg_optimizer.load_state_dict(ckpt['optimizer_state'])
    seg_scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch    = ckpt['epoch']
    best_val_dice  = ckpt['best_metric']
    log.info(f'Resumed from epoch {start_epoch}  (best val Dice: {best_val_dice:.4f})')
elif SEG_RESUME:
    log.warning('SEG_RESUME=True but no checkpoint found. Starting from scratch.')

# Training loop
log.info('Starting segmenter training …')
for epoch in range(start_epoch, SEG_NUM_EPOCHS):
    t0 = time.time()

    train_loss, train_m = seg_train_one_epoch(
        seg_model, seg_train_dl, seg_optimizer, seg_criterion, DEVICE, epoch, SEG_NUM_EPOCHS)
    val_loss, val_m = seg_evaluate(seg_model, seg_val_dl, seg_criterion, DEVICE)
    seg_scheduler.step()

    seg_train_losses.append(train_loss)
    seg_val_losses.append(val_loss)
    seg_val_dices.append(val_m['dice'])

    log.info(
        f'Epoch [{epoch+1:02d}/{SEG_NUM_EPOCHS}]  '
        f'train_loss={train_loss:.4f}  '
        f'val_loss={val_loss:.4f}  val_Dice={val_m["dice"]:.4f}  '
        f'val_IoU={val_m["iou"]:.4f}  val_Prec={val_m["precision"]:.4f}  '
        f'val_Rec={val_m["recall"]:.4f}  ({time.time()-t0:.1f}s)'
    )

    save_checkpoint({
        'epoch':           epoch + 1,
        'model_state':     seg_model.state_dict(),
        'optimizer_state': seg_optimizer.state_dict(),
        'scheduler_state': seg_scheduler.state_dict(),
        'best_metric':     best_val_dice,
    }, SEG_CHECKPOINT)

    if val_m['dice'] > best_val_dice:
        best_val_dice, no_improve = val_m['dice'], 0
        torch.save(seg_model.state_dict(), SEG_BEST)
        log.info(f'  New best val Dice: {best_val_dice:.4f} saved to {SEG_BEST}')
    else:
        no_improve += 1
        log.info(f'  No improvement ({no_improve}/{SEG_PATIENCE})')

    if no_improve >= SEG_PATIENCE:
        log.info(f'Early stopping — no improvement for {SEG_PATIENCE} epochs.')
        break

log.info('Segmenter training complete.')

2026-04-22 20:19:22,518  Found  2377 authentic  |  2751 forged
2026-04-22 20:19:22,531  Split  train=3589  val=769  test=770
2026-04-22 20:19:22,988  HTTP Request: HEAD https://huggingface.co/smp-hub/resnet34.imagenet/resolve/7a57b34f723329ff020b3f8bc41771163c519d0c/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-22 20:19:22,989  Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-22 20:19:23,007  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/smp-hub/resnet34.imagenet/7a57b34f723329ff020b3f8bc41771163c519d0c/config.json "HTTP/1.1 200 OK"
2026-04-22 20:19:23,026  HTTP Request: GET https://huggingface.co/api/resolve-cache/models/smp-hub/resnet34.imagenet/7a57b34f723329ff020b3f8bc41771163c519d0c/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

2026-04-22 20:19:23,106  HTTP Request: HEAD https://huggingface.co/smp-hub/resnet34.imagenet/resolve/7a57b34f723329ff020b3f8bc41771163c519d0c/model.safetensors "HTTP/1.1 302 Found"
2026-04-22 20:19:23,218  HTTP Request: GET https://huggingface.co/api/models/smp-hub/resnet34.imagenet/xet-read-token/7a57b34f723329ff020b3f8bc41771163c519d0c "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

2026-04-22 20:19:24,683  Starting segmenter training …
Seg epoch [01/30]: 100%|██████████████████████████| 225/225 [08:05<00:00,  2.16s/it, dice=0.3976, loss=0.4456]
2026-04-22 20:28:23,901  Epoch [01/30]  train_loss=0.6111  val_loss=0.4890  val_Dice=0.3015  val_IoU=0.1809  val_Prec=0.2571  val_Rec=0.3960  (539.2s)
2026-04-22 20:28:24,295  Checkpoint saved to /kaggle/working/segmenter_checkpoint.pth
2026-04-22 20:28:24,435    New best val Dice: 0.3015 saved to /kaggle/working/segmenter_best.pth
Seg epoch [02/30]: 100%|██████████████████████████| 225/225 [06:52<00:00,  1.83s/it, dice=0.1614, loss=0.5027]
2026-04-22 20:36:01,888  Epoch [02/30]  train_loss=0.4767  val_loss=0.4686  val_Dice=0.3448  val_IoU=0.2122  val_Prec=0.2753  val_Rec=0.4953  (457.5s)
2026-04-22 20:36:02,538  Checkpoint saved to /kaggle/working/segmenter_checkpoint.pth
2026-04-22 20:36:02,761    New best val Dice: 0.3448 saved to /kaggle/working/segmenter_best.pth
Seg epoch [03/30]: 100%|██████████████████████████| 225

---
## 8. Stage 2 — Segmenter Evaluation

In [28]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(seg_train_losses, label='Train Loss')
ax1.plot(seg_val_losses,   label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Segmenter — Loss Curves'); ax1.legend()
ax2.plot(seg_val_dices, label='Val Dice', color='green')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Dice Score')
ax2.set_title('Segmenter — Validation Dice'); ax2.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'segmenter_training_curves.png', dpi=150)
plt.show()

# Load best weights and evaluate on held-out test set
seg_model.load_state_dict(torch.load(SEG_BEST, map_location=DEVICE))
test_loss, test_m = seg_evaluate(seg_model, seg_test_dl, seg_criterion, DEVICE)

print('\n' + '='*60)
print('SEGMENTER — FINAL TEST SET RESULTS')
print('='*60)
for k, v in test_m.items():
    print(f'  {k:12s}: {v:.4f}')


SEGMENTER — FINAL TEST SET RESULTS
  dice        : 0.4410
  iou         : 0.2902
  precision   : 0.4759
  recall      : 0.4296


In [29]:
# Visual comparison: input image | ground truth mask | predicted mask
# This is the most intuitive way to judge segmentation quality
seg_model.eval()
imgs_sample, masks_sample = next(iter(seg_test_dl))
imgs_sample = imgs_sample[:6].to(DEVICE)

with torch.no_grad():
    preds_sample = (torch.sigmoid(seg_model(imgs_sample)) >= 0.5).float().cpu()

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(6, 3, figsize=(9, 18))
for i in range(6):
    # Denormalise image for display (undo ImageNet normalisation)
    img_np = np.clip(imgs_sample[i].cpu().numpy().transpose(1, 2, 0) * std + mean, 0, 1)
    axes[i, 0].imshow(img_np);                         axes[i, 0].set_title('Image')
    axes[i, 1].imshow(masks_sample[i, 0], cmap='gray'); axes[i, 1].set_title('Ground Truth')
    axes[i, 2].imshow(preds_sample[i, 0], cmap='gray'); axes[i, 2].set_title('Prediction')
    for ax in axes[i]: ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'segmenter_predictions.png', dpi=150)
plt.show()

---
## 9. Download Outputs

In [30]:
# List all output files with sizes, then provide download links
from IPython.display import FileLink, display

print('Files saved to /kaggle/working:\n')
for f in sorted(OUTPUT_DIR.glob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / 1024 / 1024
        print(f'  {f.name:45s}  {size_mb:6.1f} MB')
        display(FileLink(str(f)))

Files saved to /kaggle/working:

  gate_best.pth                                    15.6 MB


/kaggle/working/gate_best.pth

  gate_checkpoint.pth                              46.3 MB


/kaggle/working/gate_checkpoint.pth

  gate_confusion_matrix.png                         0.0 MB


...

/kaggle/working/segmenter_checkpoint.pth

  segmenter_predictions.png                         1.3 MB


/kaggle/working/segmenter_predictions.png

  segmenter_training_curves.png                     0.1 MB


/kaggle/working/segmenter_training_curves.png